# Explainable AI (XAI)

Explainable AI (XAI) refers to a set of techniques that allow us to understand, interpret, and communicate how a machine learning model makes its predictions.

In a credit scoring context, explainability is not optional:

- Credit decisions have direct financial and regulatory impact,
- Stakeholders (risk, compliance, business) need to trust and justify model outputs,
- Regulations often require transparency and reason codes for adverse decisions.

A model with good predictive performance but poor interpretability has limited business value.

## Gameplan
1. Permutation importance
   - Measure global feature importance based on performance degradation.
   - Provide a model-agnostic sanity check of which variables drive predictions.
   - Validate alignment with credit intuition and earlier EDA findings
2. SHAP – Global Explanations
    - Use SHAP values to quantify direction and magnitude of feature effects.
    - Identify:
        + Risk-increasing vs risk-reducing drivers,
        + Non-linearities and interaction patterns.
        + Produce global summaries suitable for model governance.
3. EXP-3: SHAP – Local Explanations
    - Explain individual predictions at borrower level.
    - Answer: “Why was this loan classified as high risk?”
    - Simulate real-world use cases such as manual review or customer explanation.

In [32]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [33]:
"""
1. Recreate X_test
2. Apply saved preprocess
3. XAI on calibrated model uploaded
"""

df = pd.read_csv('loans_full_schema.csv')

default_status = [
    "Charged Off",
    "In Grace Period",
    "Late (16-30 days)",
    "Late (31-120 days)"
]

non_default_status = ["Fully Paid"]

df_pd = df.loc[
    df["loan_status"].isin(default_status + non_default_status)
].copy()

# Binary target: 1 = default-like, 0 = non-default
df_pd["default"] = df_pd["loan_status"].isin(default_status).astype(int)

def compute_loan_income_ratio(df):
    income = df["annual_income"].copy()
    mask = (income.isna()) | (income <= 0)
    income[mask] = df.loc[mask, "annual_income_joint"]

    ratio = df["loan_amount"] / income
    return ratio

# Create ratio
df_pd["loan_to_income_ratio"] = compute_loan_income_ratio(df_pd)
df_pd = df_pd.loc[df_pd["loan_to_income_ratio"].notna()].copy()

# Avoid NaN in annual_income_joint
df_pd["annual_income_joint"] = df_pd["annual_income_joint"].fillna(
    df_pd["annual_income"]
)
df_pd["months_since_last_delinq"] = df_pd["months_since_last_delinq"].fillna(0)


# Selected features based on EDA + credit intuition
num_features = [
    "annual_income",
    "annual_income_joint",
    "debt_to_income",
    "emp_length",
    "earliest_credit_line",
    "total_credit_lines",
    "open_credit_lines",
    "total_debit_limit",
    "num_total_cc_accounts",
    "num_open_cc_accounts",
    "months_since_last_delinq",
    "loan_amount" , 
    "loan_to_income_ratio" # Added!
]

cat_features = [
    "homeownership",
    "verified_income",
    "public_record_bankrupt",
    "tax_liens",
    "loan_purpose",
    "application_type",
    "state",
    "emp_title"
]

features = num_features + cat_features

features = num_features + cat_features

X = df_pd[features].copy()
y = df_pd["default"].copy()


target = "default"

X = X = df_pd[num_features+cat_features].copy()
y = df_pd[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42, stratify=y
)

In [37]:
import joblib

MODEL_PATH = "credit_model_calibrated_artifact.joblib"
artifact = joblib.load(MODEL_PATH)

baseline_lr = artifact["base_model"]
platt_scaler = artifact["calibrated_model"]
preprocess = artifact["preprocess"]

num_features = artifact["num_features"]
cat_features = artifact["cat_features"]
vars_to_scale = artifact["vars_to_scale"]
vars_not_scale = artifact["vars_not_scale"]
feature_names_out = artifact["feature_names_out"]

print("Saved at:", artifact["saved_at"])
print("Calibration:", artifact["calibration_method"])

Saved at: 2026-01-21T08:41:55.221134
Calibration: platt_sigmoid


### 1: Permutation Importance**

Permutation Importance is used to measure **global feature importance** by quantifying how much the model’s predictive performance degrades when the values of a single feature are randomly shuffled.

This technique answers the question:

> *“Which variables does the model rely on most to discriminate default vs non-default?”*

Key properties:

* **Model-agnostic** (works with any classifier),
* Evaluated on a **holdout set**,
* Directly linked to **business-relevant performance metrics**.

In [ ]:
# Apply imputers (transform only) then preprocess
X_test_imp = X_test.copy()
X_test_imp[num_features] = num_imputer.transform(X_test_imp[num_features])
X_test_imp[cat_features] = cat_imputer.transform(X_test_imp[cat_features])

X_test_preprocessed = preprocess.transform(X_test_imp)
X_test_dense = X_test_preprocessed.toarray()

In [41]:
# 1) Check NaNs in raw X_test
nan_counts = X_test.isna().sum().sort_values(ascending=False)
nan_counts = nan_counts[nan_counts > 0]

print("Columns with NaNs in X_test:")
display(nan_counts)

X_test_preprocessed = preprocess.transform(X_test)
assert not np.isnan(X_test_preprocessed.toarray()).any()
print("No NaNs after preprocessing (no Pipeline)")


Columns with NaNs in X_test:


emp_title         9
emp_length        8
debt_to_income    1
dtype: int64

AssertionError: 

In [36]:
# 1. Permutation importance
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

X_test_preprocessed = preprocess.transform(X_test)


X_test_dense = X_test_preprocessed.toarray() # needed for permutation_importance()
perm_result = permutation_importance(
    estimator=platt_scaler,
    X=X_test_dense,
    y=y_test,
    scoring="roc_auc",
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)



ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:

X_test_dense = X_test_preprocessed.toarray()


perm_result = permutation_importance(
    estimator=platt_scaler,
    X=X_test_dense,
    y=y_test,
    scoring="roc_auc",
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)


ValueError: X has 408 features, but LogisticRegression is expecting 1 features as input.